# Maven CRM Sales Opportunities — audit
Score once engaged, with account, product and agent known, before Won/Lost. Open opportunities are not labelled losses. Source CSVs and the IBM notebook are never modified. Run from the repository or its notebooks directory using the existing analysis environment.

## 1. Load and inspect all five tables
The dictionary is the source for units: account revenue is **millions of USD**, not USD. Dataset reference tables are undated snapshots, so as-of-engagement availability of their values remains an assumption. Maven describes a fictitious computer-hardware company; schema realism is not evidence of real-world predictive validity.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'CONTEXT.md').is_file() and (p / 'datasets').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Run inside the cloned repository with datasets available.')
DATA = ROOT / 'datasets' / 'CRM+Sales+Opportunities'
names = ['accounts', 'products', 'sales_pipeline', 'sales_teams', 'data_dictionary']
tables = {n: pd.read_csv(DATA / (n + '.csv'), keep_default_na=False, na_values=['']) for n in names}
pd.set_option('display.max_columns', 24)
pd.set_option('display.max_rows', 45)
pd.set_option('display.max_colwidth', 90)
audit_summary = pd.DataFrame([{'table': n, 'rows': len(t), 'columns': len(t.columns), 'duplicate_rows': int(t.duplicated().sum()), 'missing_cells': int(t.isna().sum().sum())} for n,t in tables.items()]).set_index('table')
display(audit_summary)
display(tables['data_dictionary'])
column_audit = pd.concat([pd.DataFrame({'table': n, 'column': t.columns, 'dtype': t.dtypes.astype(str).values, 'missing': t.isna().sum().values, 'unique': t.nunique().values, 'examples': [', '.join(map(str,t[c].dropna().unique()[:3])) for c in t]}) for n,t in tables.items()], ignore_index=True)
display(column_audit)
print('Input SHA256:', {n: hashlib.sha256((DATA / (n+'.csv')).read_bytes()).hexdigest() for n in names})

,rows,columns,duplicate_rows,missing_cells
table,,,,
accounts,85,7,0,70
products,7,3,0,0
sales_pipeline,8800,8,0,6103
sales_teams,35,3,0,0
data_dictionary,21,3,0,0


,Table,Field,Description
0,accounts,account,Company name
1,accounts,sector,Industry
2,accounts,year_established,Year Established
3,accounts,revenue,Annual revenue (in millions of USD)
4,accounts,employees,Number of employees
5,accounts,office_location,Headquarters
6,accounts,subsidiary_of,Parent company
7,products,product,Product name
8,products,series,Product series
9,products,sales_price,Suggested retail price


,table,column,dtype,missing,unique,examples
0,accounts,account,str,0,85,"Acme Corporation, Betasoloin, Betatech"
1,accounts,sector,str,0,10,"technolgy, medical, retail"
2,accounts,year_established,int64,0,35,"1996, 1999, 1986"
3,accounts,revenue,float64,0,85,"1100.04, 251.41, 647.18"
4,accounts,employees,int64,0,85,"2822, 495, 1185"
5,accounts,office_location,str,0,15,"United States, Kenya, Philipines"
6,accounts,subsidiary_of,str,70,7,"Acme Corporation, Massive Dynamic, Bubba Gump"
7,products,product,str,0,7,"GTX Basic, GTX Pro, MG Special"
8,products,series,str,0,3,"GTX, MG, GTK"
9,products,sales_price,int64,0,7,"550, 4821, 55"


Input SHA256: {'accounts': 'e5242324768a563fc632cddfed49a29acbbf2892b8a3c6453cc9650de9ae0358', 'products': 'b461c98739749ee97abd405a121c45f0cdc38e1d1ec2946a7005ecda08a2d31a', 'sales_pipeline': '825ce8f6c32d4009548b468df3173d55a46fd73f2531f532c5459371dc52adf2', 'sales_teams': 'aeff1272ebe196f5a27e3fc0578aa27abf48ed9ae461aa344fb95990e5ad8bd1', 'data_dictionary': '22b34e498d07e3d7f322afdbf81d70a5dc0a389792944e50ca2af86a3597f0af'}


## 2. Keys, normalization and join coverage
Normalize GTXPro only when whitespace removal gives a unique reference match. Reference-key uniqueness and many-to-one joins prevent row multiplication. Blank subsidiary_of means no recorded parent. Other spelling differences are retained rather than guessed.

In [2]:
accounts, products, pipeline, teams = [tables[n].drop_duplicates().copy() for n in ['accounts','products','sales_pipeline','sales_teams']]
for table,key in [(accounts,'account'),(products,'product'),(teams,'sales_agent')]:
    assert table[key].notna().all() and table[key].is_unique
assert pipeline.opportunity_id.notna().all() and pipeline.opportunity_id.is_unique
references = {'account': accounts, 'product': products, 'sales_agent': teams}
for key,reference in references.items():
    print(key, 'unmatched before:', sorted(set(pipeline[key].dropna())-set(reference[key])))
product_map = {}
for value in set(pipeline['product'].dropna())-set(products['product']):
    matches = [p for p in products['product'] if ''.join(p.split()) == ''.join(value.split())]
    if len(matches) == 1:
        product_map[value] = matches[0]
assert product_map == {'GTXPro':'GTX Pro'}, f'Review normalization: {product_map}'
print('Verified mapping:', product_map, '| affected rows:', pipeline['product'].isin(product_map).sum())
pipeline['product'] = pipeline['product'].replace(product_map)
join_coverage = pd.DataFrame([{'key':key, 'pipeline_unique':pipeline[key].nunique(), 'reference_unique':reference[key].nunique(), 'missing_rows':pipeline[key].isna().sum(), 'unmatched_non_null':(pipeline[key].notna() & ~pipeline[key].isin(reference[key])).sum(), 'matched_rows':pipeline[key].isin(reference[key]).sum(), 'unused_reference_keys':len(set(reference[key])-set(pipeline[key].dropna()))} for key,reference in references.items()])
display(join_coverage)
print('Parent names absent from accounts:', sorted(set(accounts.subsidiary_of.dropna())-set(accounts.account)))
joined = pipeline.copy()
for key,reference in references.items():
    joined = joined.merge(reference,on=key,how='left',validate='many_to_one',indicator=key+'_join')
assert len(joined) == len(pipeline)
for date in ['engage_date','close_date']:
    parsed = pd.to_datetime(joined[date],format='%Y-%m-%d',errors='coerce')
    assert not (joined[date].notna() & parsed.isna()).any()
    joined[date] = parsed
for col in ['sector','office_location','series','manager','regional_office']:
    print(col, sorted(joined[col].dropna().unique()))

account unmatched before: []
product unmatched before: ['GTXPro']
sales_agent unmatched before: []
Verified mapping: {'GTXPro': 'GTX Pro'} | affected rows: 1480


,key,pipeline_unique,reference_unique,missing_rows,unmatched_non_null,matched_rows,unused_reference_keys
0,account,85,85,1425,0,7375,0
1,product,7,7,0,0,8800,0
2,sales_agent,30,35,0,0,8800,5


Parent names absent from accounts: []
sector ['employment', 'entertainment', 'finance', 'marketing', 'medical', 'retail', 'services', 'software', 'technolgy', 'telecommunications']
office_location ['Belgium', 'Brazil', 'China', 'Germany', 'Italy', 'Japan', 'Jordan', 'Kenya', 'Korea', 'Norway', 'Panama', 'Philipines', 'Poland', 'Romania', 'United States']
series ['GTK', 'GTX', 'MG']
manager ['Cara Losch', 'Celia Rouche', 'Dustin Brinkmann', 'Melvin Marxen', 'Rocco Neubert', 'Summer Sewald']
regional_office ['Central', 'East', 'West']


## 3. Supervised population and open opportunities
Won=1; Lost=0. Require valid joins and an engagement date before or on the closing date. Keep open opportunities separately; missing outcomes are never replaced with losses. Later engagement cohorts may be biased toward fast-closing deals because follow-up ends with the extract.

In [3]:
assert set(joined.deal_stage) <= {'Won','Lost','Engaging','Prospecting'}
display(joined.deal_stage.value_counts().rename('opportunities').to_frame())
is_closed = joined.deal_stage.isin(['Won','Lost'])
valid_keys = joined[['account_join','product_join','sales_agent_join']].eq('both').all(axis=1)
valid_contract = valid_keys & joined.engage_date.notna()
valid_dates = joined.close_date.notna() & (joined.close_date >= joined.engage_date)
closed = joined.loc[is_closed & valid_contract & valid_dates].copy()
closed['target'] = closed.deal_stage.map({'Won':1,'Lost':0}).astype(int)
open_opportunities = joined.loc[~is_closed].copy()
open_opportunities['scoreable_at_engagement'] = valid_contract.loc[open_opportunities.index]
print('Raw closed:', is_closed.sum(), '| usable closed:', len(closed), '| excluded:', is_closed.sum()-len(closed))
print('Closed missing dates:', joined.loc[is_closed,['engage_date','close_date']].isna().sum().to_dict())
print('Close before engagement:', (joined.close_date < joined.engage_date).sum())
print('Open:', len(open_opportunities), '| currently contract-eligible:', open_opportunities.scoreable_at_engagement.sum())
print('Unique usable entities:', closed[['account','product','sales_agent']].nunique().to_dict())
display(closed.groupby('deal_stage').agg(opportunities=('target','size')).assign(rate=lambda x:x.opportunities/x.opportunities.sum()))
display(joined.groupby('deal_stage').agg(engage_min=('engage_date','min'),engage_max=('engage_date','max'),close_min=('close_date','min'),close_max=('close_date','max')))
display(joined.assign(month=joined.engage_date.dt.to_period('M').astype(str)).groupby(['month','deal_stage']).size().unstack(fill_value=0))
display(closed.groupby(closed.close_date.dt.to_period('M')).agg(closed=('target','size'),won_rate=('target','mean')))
print('Founding year after engagement:', (closed.year_established > closed.engage_date.dt.year).sum())
print('Nonpositive financial fields:', {c:int(closed[c].le(0).sum()) for c in ['revenue','employees','sales_price']})

,opportunities
deal_stage,
Won,4238
Lost,2473
Engaging,1589
Prospecting,500


Raw closed: 6711 | usable closed: 6711 | excluded: 0
Closed missing dates: {'engage_date': 0, 'close_date': 0}
Close before engagement: 0
Open: 2089 | currently contract-eligible: 501
Unique usable entities: {'account': 85, 'product': 7, 'sales_agent': 30}


,opportunities,rate
deal_stage,,
Lost,2473,0.368499
Won,4238,0.631501


,engage_min,engage_max,close_min,close_max
deal_stage,,,,
Engaging,2016-11-03,2017-12-22,NaT,NaT
Lost,2016-11-04,2017-12-22,2017-03-01,2017-12-31
Prospecting,NaT,NaT,NaT,NaT
Won,2016-10-20,2017-12-27,2017-03-01,2017-12-31


deal_stage,Won,Engaging,Lost
month,,,
2016-10,7,0,0
2016-11,83,13,19
2016-12,160,40,36
2017-01,181,65,66
2017-02,277,95,143
2017-03,440,137,215
2017-04,427,121,326
2017-05,366,134,222
2017-06,453,111,228


,closed,won_rate
close_date,,
2017-03,647,0.820711
2017-04,586,0.486348
2017-05,805,0.544099
2017-06,641,0.828393
2017-07,627,0.491228
2017-08,785,0.568153
2017-09,635,0.792126
2017-10,566,0.492933
2017-11,768,0.528646


Founding year after engagement: 5
Nonpositive financial fields: {'revenue': 0, 'employees': 0, 'sales_price': 0}


## Audit conclusions and feature policy
8,800 unique opportunities; 6,711 closed (4,238 Won, 2,473 Lost), 2,089 open. All closed rows have valid keys/dates. The reference-supported GTXPro → GTX Pro mapping affects 1,480 rows; source files are not edited. There are 35 reference agents, but only 30 appear in the pipeline. Account revenue and employees each uniquely identify all 85 reference accounts; list price uniquely identifies each of seven products. Removing names therefore does **not** prove entity-independent generalization.

Five closed rows have founding years later than their engagement year. Preserve these records but make negative company age missing and flag the inconsistency for review, rather than claiming a negative age.

**Safe candidates, conditional on undated snapshots being available at engagement:** account sector/financials, founding year, recorded parent, product metadata and initial agent assignment; engagement date defines scoring time.
**Engineered:** age/logs/ratios/calendar, and outcomes closed strictly before engagement. An outcome is assumed observable on its close date; no ingestion timestamps are supplied.
**Questionable:** names, exact price (product proxy), exact account financials (account proxies), geography and undated team assignments. Test identity and coarsened-input diagnostics.
**Exclude from inputs:** current deal_stage, close_date, close_value, opportunity_id, and engagement-to-close duration. Open outcomes are unknown, not losses.

[Maven source](https://mavenanalytics.io/data-playground/crm-sales-opportunities): fictitious computer hardware company. The local dictionary documents meaning, not the data-generation process, effective dates or license. Confirm download terms before redistribution. Closed-only evaluation remains subject to unresolved-deal selection bias.